In [80]:
import numpy as np
import pandas as pd
import  matplotlib.pyplot as plt
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')
import plotly.express as px 

from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold,GridSearchCV,train_test_split
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model    import LogisticRegression
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

In [2]:
df=pd.read_csv("Travel.csv")

In [5]:
df.head()

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,200000,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3,3.0,Deluxe,3.0,Single,1.0,1,2,1,0.0,Manager,20993.0
1,200001,0,49.0,Company Invited,1,14.0,Salaried,Male,3,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0
2,200002,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,4.0,Basic,3.0,Single,7.0,1,3,0,0.0,Executive,17090.0
3,200003,0,33.0,Company Invited,1,9.0,Salaried,Female,2,3.0,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0
4,200004,0,NaN,Self Enquiry,1,8.0,Small Business,Male,2,3.0,Basic,4.0,Divorced,1.0,0,5,1,0.0,Executive,18468.0


In [7]:
df.isnull().sum()

CustomerID                    0
ProdTaken                     0
Age                         226
TypeofContact                25
CityTier                      0
DurationOfPitch             251
Occupation                    0
Gender                        0
NumberOfPersonVisiting        0
NumberOfFollowups            45
ProductPitched                0
PreferredPropertyStar        26
MaritalStatus                 0
NumberOfTrips               140
Passport                      0
PitchSatisfactionScore        0
OwnCar                        0
NumberOfChildrenVisiting     66
Designation                   0
MonthlyIncome               233
dtype: int64

In [9]:
df['ProdTaken']=np.where(df['ProdTaken']==1,'Yes','No')

In [11]:
df.head()

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,200000,Yes,41.0,Self Enquiry,3,6.0,Salaried,Female,3,3.0,Deluxe,3.0,Single,1.0,1,2,1,0.0,Manager,20993.0
1,200001,No,49.0,Company Invited,1,14.0,Salaried,Male,3,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0
2,200002,Yes,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,4.0,Basic,3.0,Single,7.0,1,3,0,0.0,Executive,17090.0
3,200003,No,33.0,Company Invited,1,9.0,Salaried,Female,2,3.0,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0
4,200004,No,NaN,Self Enquiry,1,8.0,Small Business,Male,2,3.0,Basic,4.0,Divorced,1.0,0,5,1,0.0,Executive,18468.0


In [13]:
x=df.drop(['CustomerID','ProdTaken'],axis=1)

In [15]:
x.head()

,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,41.0,Self Enquiry,3,6.0,Salaried,Female,3,3.0,Deluxe,3.0,Single,1.0,1,2,1,0.0,Manager,20993.0
1,49.0,Company Invited,1,14.0,Salaried,Male,3,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0
2,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,4.0,Basic,3.0,Single,7.0,1,3,0,0.0,Executive,17090.0
3,33.0,Company Invited,1,9.0,Salaried,Female,2,3.0,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0
4,NaN,Self Enquiry,1,8.0,Small Business,Male,2,3.0,Basic,4.0,Divorced,1.0,0,5,1,0.0,Executive,18468.0


In [17]:
y=df['ProdTaken']

In [19]:
numeric_columns=[col for col in x.columns if x[col].dtypes!='O']

In [21]:
numeric_columns

['Age',
 'CityTier',
 'DurationOfPitch',
 'NumberOfPersonVisiting',
 'NumberOfFollowups',
 'PreferredPropertyStar',
 'NumberOfTrips',
 'Passport',
 'PitchSatisfactionScore',
 'OwnCar',
 'NumberOfChildrenVisiting',
 'MonthlyIncome']

In [23]:
categorical_columns=[col for col in x.columns if x[col].dtypes=='O']

In [25]:
categorical_columns

['TypeofContact',
 'Occupation',
 'Gender',
 'ProductPitched',
 'MaritalStatus',
 'Designation']

In [27]:
numeric_pipeline=Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())])

In [47]:
categorical_pipeline=Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))])

In [49]:
preprocessor=ColumnTransformer([('num',numeric_pipeline,numeric_columns),('cat',categorical_pipeline,categorical_columns)])

In [102]:
dt_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model",DecisionTreeClassifier())
])

In [104]:
knn_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model",KNeighborsClassifier())
])

In [106]:
logis_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model",LogisticRegression())
])

In [108]:
svc_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model",SVC())
])

In [110]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)


In [112]:
from sklearn.model_selection import cross_val_score

models = {
    "Logistic Regression": logis_pipeline,
    "SVC": svc_pipeline,
    "KNN": knn_pipeline,
    "Decision Tree": dt_pipeline
}

for name, model in models.items():

    scores = cross_val_score(
        model,
        x,
        y,
        cv=5,
        scoring="accuracy",
        n_jobs=-1
    )

    print(name)
    print("Scores:", scores)
    print("Mean Accuracy:", scores.mean())
    print()

Logistic Regression
Scores: [0.8394683  0.84764826 0.84560327 0.84032753 0.83930399]
Mean Accuracy: 0.8424702722955167

SVC
Scores: [0.87218814 0.86912065 0.87423313 0.8741044  0.87717503]
Mean Accuracy: 0.8733642698214348

KNN
Scores: [0.9202454  0.91002045 0.91411043 0.91197544 0.93654043]
Mean Accuracy: 0.9185784286022276

Decision Tree
Scores: [0.94580777 0.90184049 0.94171779 0.89355169 0.93654043]
Mean Accuracy: 0.9238916343801087



In [98]:
y_pred=model_pipeline.predict(x_test)

In [100]:
acc=accuracy_score(y_test,y_pred)
print(acc*100)

91.1042944785276
